# 04. Distribution centers: re-randomization with few units

**Sector:** logistics. **Decision:** does a new routing engine improve the
on-time delivery rate? The units are **only 24 distribution centers (DCs)**, and
they differ in volume (`throughput_pre`) and number of docks (`dock_count`). With
so few units, a single draw can come out **imbalanced by bad luck**,
concentrating the large DCs in one arm.

The fix is to **re-randomize**: draw repeatedly until the covariates are balanced
(Mahalanobis criterion), then use a **randomization inference that respects that
same criterion**. Theory:
[II. Designs](../../../docs/en/theory/02-designs.md) (topic 6) and
[IV. Inference](../../../docs/en/theory/04-inference.md) (topic 13).


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd


def _find_data():
    """Locate examples/use_cases/data whether run from the notebook or the root."""
    for base in [Path.cwd(), *Path.cwd().parents]:
        for cand in (base / "data", base / "examples" / "use_cases" / "data"):
            if (cand / "ecommerce_checkout.csv").exists():
                return cand
    raise FileNotFoundError("Could not locate examples/use_cases/data")


DATA = _find_data()

df = pd.read_csv(DATA / "logistics_dc.csv")
print(df.shape, "distribution centers")
df.head()


## 1. A plain CRD can be imbalanced

First, an ordinary `CRD` draw and the covariate balance. With 24 units, it is not
unusual for the SMD of `throughput_pre` or `dock_count` to exceed `0.1`.


In [ ]:
from skxperiments.design.crd import CRD
from skxperiments.design.balance import check_balance

crd = CRD(p=0.5, seed=404).randomize(df[["region", "throughput_pre", "dock_count"]].copy())
check_balance(crd)[["covariate", "smd"]].round(3)


## 2. Re-randomize until balanced

`ReRandomizedCRD` accepts only draws whose Mahalanobis distance falls below a
threshold. We use `chi2.ppf(0.10, df=2)` (accepts the ~10% most balanced draws;
with small `n`, a not-too-strict threshold avoids excessive rejection). Balance
afterward should be visibly better.


In [ ]:
from scipy.stats import chi2

from skxperiments.design.rerandomized_crd import ReRandomizedCRD
from skxperiments.diagnostics import BalanceReport

threshold = float(chi2.ppf(0.10, df=2))
design = ReRandomizedCRD(
    covariates=["throughput_pre", "dock_count"], threshold=threshold,
    p=0.5, seed=404, max_attempts=20000,
)
assignment = design.randomize(df[["region", "throughput_pre", "dock_count"]].copy())
report = BalanceReport().run(assignment)
print("imbalanced covariates:", report.imbalanced)


In [ ]:
from skxperiments.reporting import plot_balance

ax = plot_balance(report)
ax.set_title("Balance after re-randomization")
ax.figure


## 3. Inference that respects the criterion

We attach the outcome (on-time rate) and analyze. `RandomizationTest`
re-randomizes by the **same** mechanism as the design (via `draw()`), so the
permutations also obey the re-randomization criterion, keeping the test valid
(not conservative). With 24 units, `C(24,12)` is huge, so the p-value comes by
Monte Carlo. We compare with the Neyman interval.


In [ ]:
from skxperiments.core.assignment import CRDAssignment
from skxperiments.estimators.difference_in_means import DifferenceInMeans
from skxperiments.inference import RandomizationTest, NeymanCI

t = assignment.data_[assignment.treatment_col_].to_numpy()
data = assignment.data_.copy()
data["on_time"] = np.where(t == 1, df["on_time_y1"].to_numpy(), df["on_time_y0"].to_numpy())
assignment = CRDAssignment(
    data=data, treatment_col=assignment.treatment_col_, design=design, seed=404
)

rand = RandomizationTest(
    estimator=DifferenceInMeans(outcome_col="on_time"), n_permutations=2000, seed=0
).fit(assignment).estimate()
neyman = NeymanCI(estimator=DifferenceInMeans(outcome_col="on_time")).fit(assignment).estimate()

print(f"Randomization test: ATE={rand.ate:.3f} pp, p={rand.p_value:.4f}")
print(f"Neyman CI:          ATE={neyman.ate:.3f} pp, "
      f"CI=({neyman.ci[0]:.3f}, {neyman.ci[1]:.3f})")


In [ ]:
from skxperiments.reporting import plot_null_distribution

ax = plot_null_distribution(rand)
ax.set_title("Randomization null distribution")
ax.figure


## Decision

The true effect is `+1.5` percentage points in the on-time rate. The estimate
lands close to it and the randomization test rejects the null. The design lesson:
with few large units, re-randomization protects against an unlucky draw, and the
inference **must** honor the criterion so it does not become conservative. A plain
t test here would be valid but less powerful.

Next step: [05. Many metrics in streaming](05_streaming_many_metrics.ipynb).
